# ST-GCN - combined regularization test

Same paper-faithful architecture and schedule as the winning advanced-augmentation run (70.72% test top-1, 21.16pp gap), with three new changes stacked together:

1. **`tcn_dropout=0.25`** added to the backbone - deliberately breaking from the paper's either/or design (augmentation replaces dropout). That design was calibrated for Kinetics (240,000 clips); our training set is ~6,255 clips, nearly 40x smaller. Worth testing whether a dataset this much smaller benefits from both regularizers at once, at a modest 0.25 rather than the paper's full 0.5 since augmentation is already doing real work.
2. **`label_smoothing=0.1`** on the loss - untested until now, cheap, standard fix for exactly this train/test gap pattern.
3. **Augmentation probability raised to 0.8** (from the default 0.5) - the earlier always-on test was run on the broken 10-stage architecture and told us nothing reliable about this one; genuinely retesting here.

Also `weight_decay=1e-3` (up from 5e-4) - modest bump, combined rather than isolated since it's cheap to include alongside the others.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import functools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score, confusion_matrix

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(42)
np.random.seed(42)

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### paths

In [2]:
OUT_ROOT = "/workspace/outputs/stgcn_paper_faithful_combined_reg"
CKPT_ROOT = "/workspace/checkpoints/stgcn_paper_faithful_combined_reg"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - advanced augmentation, probability raised to 0.8

Registering a new policy in dataset.py's live registry (a module-level dict, mutable at runtime) rather than editing the file - keeps this experiment self-contained.

In [3]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

ds.AUGMENTATION_POLICIES["advanced_p08"] = [
    functools.partial(fn, p=0.8) for fn in ds.ADVANCED_AUGMENTATIONS
]

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced_p08")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=42)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### batch -> ST-GCN input - same verified shape function

In [4]:
def batch_to_stgcn_input(batch):
    kp = batch["keypoints"].float()
    x = kp.unsqueeze(1).unsqueeze(1)   # (N, num_clips=1, M=1, T, V, C)
    return x


### model config - same 9-stage architecture, now with tcn_dropout added

Everything else identical to the winning advanced run - only the dropout line is new.

In [5]:
model_cfg = dict(
    type="RecognizerGCN",
    backbone=dict(
        type="STGCN",
        graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
        in_channels=3,
        base_channels=64,
        ch_ratio=2,
        num_stages=9,
        inflate_stages=[4, 7],
        down_stages=[4, 7],
        tcn_dropout=0.25,   # NEW - deliberately combined with augmentation, see markdown above
    ),
    cls_head=dict(
        type="GCNHead",
        num_classes=num_classes,
        in_channels=256,
        loss_cls=dict(type="CrossEntropyLoss"),
    ),
)

model = MODELS.build(model_cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model built, {n_params:,} parameters")


model built, 3,027,379 parameters


### SMOKE TEST - run this cell alone first

The backbone changed (tcn_dropout added), so re-verify the shape contract before trusting anything below - don't assume the earlier verification carries over unchanged.

In [6]:
smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_stgcn_input(smoke_batch).to(device)
print("input shape:", smoke_input.shape)

model.eval()
with torch.no_grad():
    smoke_output = model(smoke_input, mode="tensor", stage="head")

print("output shape:", smoke_output.shape)
assert smoke_output.shape == (4, num_classes), f"expected (4, {num_classes}), got {smoke_output.shape}"
print("smoke test passed")


input shape: torch.Size([4, 1, 1, 30, 17, 3])
output shape: torch.Size([4, 18])
smoke test passed


### metrics - identical definitions used throughout this project

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    top1 = accuracy_score(labels, preds)
    top5 = top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes)))
    mca = balanced_accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    return {"top1": top1, "top5": top5, "mca": mca, "macro_f1": macro_f1, "labels": labels, "preds": preds}


### training - same schedule as before, label smoothing added to the loss

In [8]:
NUM_EPOCHS = 60
PATIENCE = 20
BATCH_SIZE = 16
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-3   # up from 5e-4 in the previous run - modest bump, tested combined not isolated
LABEL_SMOOTHING = 0.1

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=balanced_sampler, num_workers=4, pin_memory=True, persistent_workers=True)

task = Task.init(project_name="BadmintonPoseAnalysis", task_name="stgcn_paper_faithful_combined_reg", reuse_last_task_id=False)
logger = task.get_logger()
task.connect({
    "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
    "num_stages": 9, "inflate_stages": [4, 7], "down_stages": [4, 7],
    "tcn_dropout": 0.25, "label_smoothing": LABEL_SMOOTHING,
    "augmentation_policy": "advanced_p08", "ntu60_pretrained": False,
})

optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_mca": [], "lr": []}
best_val_top1 = 0.0
epochs_without_improvement = 0
best_state = None

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_labels, epoch_preds = [], []

    for batch in train_loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(x, mode="tensor", stage="head")
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * x.size(0)
        epoch_labels.append(y.cpu().numpy())
        epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

    scheduler.step()

    train_loss = epoch_loss / len(train_loader.dataset)
    train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))

    val_metrics = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_metrics["top1"])
    history["val_mca"].append(val_metrics["mca"])
    history["lr"].append(optimizer.param_groups[0]["lr"])

    logger.report_scalar("loss", "train", train_loss, iteration=epoch)
    logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
    logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
    logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

    if val_metrics["top1"] > best_val_top1:
        best_val_top1 = val_metrics["top1"]
        epochs_without_improvement = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"epoch {epoch:3d}  lr={lr_now:.5f}  train_loss={train_loss:.4f}  "
          f"train_acc={train_acc:.4f}  val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f}")
        break

model.load_state_dict(best_state)


ClearML Task: created new task id=98a6aa3b888b42d786a4fa4ead2c48f6
2026-09-10 04:40:18,226 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/98a6aa3b888b42d786a4fa4ead2c48f6/output/log
epoch   0  lr=0.01000  train_loss=3.9118  train_acc=0.2732  val_top1=0.5173  val_mca=0.3798
epoch   1  lr=0.01000  train_loss=3.3016  train_acc=0.4216  val_top1=0.5851  val_mca=0.5179
epoch   2  lr=0.01000  train_loss=3.0149  train_acc=0.4921  val_top1=0.5723  val_mca=0.5418
epoch   3  lr=0.01000  train_loss=2.8233  train_acc=0.5423  val_top1=0.5915  val_mca=0.5454
epoch   4  lr=0.01000  train_loss=2.6971  train_acc=0.5750  val_top1=0.6261  val_mca=0.5784
epoch   5  lr=0.01000  train_loss=2.6007  train_acc=0.5931  val_top1=0.6415  val_mca=0.5690
epoch   6  lr=0.01000  train_loss=2.5258  train_acc=0.6129  val_top1=0.6594  val_mca=0.5752
epoch   7  lr=0.01000  train_loss=2.4633  tr

<All keys matched successfully>

███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  7.77MB/s]: 


### final evaluation and comparison

In [9]:
LI_ET_AL_BENCHMARK = {"test_top1": 0.7441, "test_top5": 0.9376, "test_mca": 0.6144}
PREVIOUS_ADVANCED_RUN = {"test_top1": 0.7072, "gap": 0.2116, "test_top5": 0.9335, "test_mca": 0.6155, "test_macro_f1": 0.6309}

train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
val_final = evaluate(model, val_loader)
test_final = evaluate(model, test_loader)

metrics = {
    "train_acc": train_final["top1"],
    "val_acc": val_final["top1"],
    "test_top1": test_final["top1"],
    "test_top5": test_final["top5"],
    "test_mca": test_final["mca"],
    "test_macro_f1": test_final["macro_f1"],
    "train_test_gap": train_final["top1"] - test_final["top1"],
}

ckpt_path = os.path.join(CKPT_ROOT, "stgcn_combined_reg_best.pt")
torch.save(model.state_dict(), ckpt_path)

with open(os.path.join(OUT_ROOT, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
with open(os.path.join(OUT_ROOT, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

task.upload_artifact("checkpoint", ckpt_path)
task.upload_artifact("metrics", metrics)
task.close()

print()
print(f"train_acc={metrics['train_acc']:.4f}  test_top1={metrics['test_top1']:.4f}  gap={metrics['train_test_gap']:.4f}")
print(f"test_top5={metrics['test_top5']:.4f}  test_mca={metrics['test_mca']:.4f}  test_macro_f1={metrics['test_macro_f1']:.4f}")
print()
print(f"previous (advanced, no extra reg): test_top1={PREVIOUS_ADVANCED_RUN['test_top1']:.4f}  gap={PREVIOUS_ADVANCED_RUN['gap']:.4f}")
print(f"Li et al. benchmark:                test_top1={LI_ET_AL_BENCHMARK['test_top1']:.4f}  test_top5={LI_ET_AL_BENCHMARK['test_top5']:.4f}  test_mca={LI_ET_AL_BENCHMARK['test_mca']:.4f}")



train_acc=0.8864  test_top1=0.6969  gap=0.1895
test_top5=0.9412  test_mca=0.6089  test_macro_f1=0.6097

previous (advanced, no extra reg): test_top1=0.7072  gap=0.2116
Li et al. benchmark:                test_top1=0.7441  test_top5=0.9376  test_mca=0.6144


### training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"]); axes[0].set_title("Train loss")
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
axes[2].plot(history["val_mca"]); axes[2].set_title("Val Mean Class Accuracy")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
plt.savefig(os.path.join(OUT_ROOT, "training_curves.png"), dpi=200)
plt.show()


---

Compare `train_test_gap` and `test_top1` directly against the printed previous-run numbers above. If the gap closed further without top-1 dropping, the combined regularization is genuinely helping. If top-1 drops even as the gap closes, that's a sign one of the three changes (most likely tcn_dropout=0.25, the most aggressive of the three) is now under-fitting rather than just regularizing - worth backing off that specific knob first before concluding the whole combination doesn't work.